# From Tickets to a Reliable Summary

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_11/materials/notebooks/07_aggregation_validation.ipynb)

[View on GitHub](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_11/materials/notebooks/07_aggregation_validation.ipynb)

**Language and environment:** Python 3 represents MongoDB aggregation pipelines
as lists of dictionaries. Run in Colab or local Jupyter with `mongomock` or your
own Atlas cluster.

A manager wants the number of active requests in each category. A pipeline can
answer that question, but an innocent-looking array operation can inflate the
counts. Today we will see the error and explain it. The validation reference material is grouped below as optional reference; it is not needed for this lab.

This is a fresh four-ticket teaching case, not the complete 12-ticket CSV dataset.
You do not need Week 10's database. The default runs aggregation locally with
`mongomock`. Today, run the aggregation and explanation sections, then submit your saved copy’s Colab link.

**Validation is not needed for this lab. It remains optional reference; no next-week assignment is implied. Today, finish the aggregation work and submit your saved Colab copy’s sharing link.**

## Complete data reference


## One document means one request

This independent four-ticket fixture is not the twelve-ticket CSV or Week 10 fixture. All application fields and events are listed below. MongoDB adds a generated `_id`; `ticket_id` is a separate unique course ID. `assignee_id` refers conceptually to a staff member, but no staff collection is joined in this exercise. The field is an integer or explicit null: ticket 1004 is unassigned. A missing field would be different from this stored null.

`category`, `status`, `priority`, and `subject` are text. Category is streetlight or sanitation; status here is new/open/in_progress/resolved; the active filter includes new/open/in_progress. Priority is urgent/high/low here (medium also exists elsewhere in the course). `opened_at` is a UTC BSON date, represented below as an ISO string for readability. `events` is an array of embedded documents; `events.type` is text. Ticket 1004's empty array is present, not null or missing. No event time or actor field exists in this fixture.

```json
[
  {
    "ticket_id": 1001,
    "category": "streetlight",
    "status": "open",
    "priority": "urgent",
    "subject": "Dark streetlight",
    "assignee_id": 201,
    "opened_at": "2026-02-01T00:00:00+00:00",
    "events": [
      {
        "type": "created"
      },
      {
        "type": "assigned"
      }
    ]
  },
  {
    "ticket_id": 1002,
    "category": "sanitation",
    "status": "in_progress",
    "priority": "high",
    "subject": "Missed pickup",
    "assignee_id": 202,
    "opened_at": "2026-02-02T00:00:00+00:00",
    "events": [
      {
        "type": "created"
      }
    ]
  },
  {
    "ticket_id": 1003,
    "category": "streetlight",
    "status": "resolved",
    "priority": "urgent",
    "subject": "Lamp repaired",
    "assignee_id": 201,
    "opened_at": "2026-02-03T00:00:00+00:00",
    "events": [
      {
        "type": "created"
      },
      {
        "type": "resolved"
      }
    ]
  },
  {
    "ticket_id": 1004,
    "category": "streetlight",
    "status": "new",
    "priority": "low",
    "subject": "Flickering lamp",
    "assignee_id": null,
    "opened_at": "2026-02-04T00:00:00+00:00",
    "events": []
  }
]
```

## Grain check

Active ticket IDs are 1001, 1002, 1004. Default unwind emits event rows for 1001 twice and 1002 once; 1004 emits none. Both totals are 3, but they describe different populations. Group before unwind when the question counts tickets.


## Connection or Local Practice

Keep `USE_ATLAS = False` to begin without an account. Package installation still
requires internet access. For Atlas, use your own existing Free cluster and set
the switch to `True`. The next cell prints a new practice database name and this
runtime's public IPv4 address. It contacts ipify without database credentials.

Colab runs on Google's computer, not your laptop. **Add Current IP** in your
laptop's browser can therefore add the wrong address. Use the printed `/32`
address, which allows one IPv4 address. Do not open access from everywhere.

In [ ]:
#@title Cell 01 — Install practice packages
%pip -q install "pymongo>=4.13,<5" "mongomock>=4.3,<5"

In [ ]:
#@title Cell 02a — Environment choice (keep visible)
USE_ATLAS = False  # Required path: local practice; Week 12 uses actual Atlas.


In [ ]:
#@title Cell 02b — Supplied imports and isolated run name
from datetime import datetime, timezone
from getpass import getpass
from ipaddress import IPv4Address
from pprint import pprint
from urllib.request import urlopen
from uuid import uuid4

import mongomock
from pymongo import MongoClient
from pymongo.errors import OperationFailure
from pymongo.server_api import ServerApi

database_name = "cst4714_pipeline_" + uuid4().hex[:8]
print("Practice database:", database_name)

if USE_ATLAS:
    try:
        with urlopen("https://api.ipify.org", timeout=10) as response:
            runtime_ip = str(IPv4Address(response.read().decode().strip()))
        print("Temporary Atlas IP access-list entry:", runtime_ip + "/32")
    except Exception:
        raise RuntimeError(
            "The runtime IP check failed. Retry or use local mode; "
            "do not open access from everywhere."
        ) from None

**Atlas only: pause before connecting.** In **Network Access / IP Access List**,
add the printed address as a temporary entry and wait for it to become active.
Recheck after a runtime restart or an outgoing IP change. Your database user is
different from your Atlas website login. It needs read/write access to this
practice database and permission to modify its collection validator (`collMod`).
Ask the instructor to help scope that permission rather than changing a shared
collection or disabling validation. Local mode remains available.

In **Connect > Drivers**, select Python and copy the driver URI. Replace its
password placeholder; reserved characters in the password need URI percent
encoding. Enter the finished URI only in the hidden prompt, never a code cell.
`ping` tests whether MongoDB responds, not whether every later operation is
authorized. Keep TLS verification enabled. Check the runtime IP, database user,
URI, deployment state, and current driver if connection fails.

Run setup once per experiment. Clean up before creating a new practice name.
These handles select a database and collection; the first write stores data.

In [ ]:
#@title Cell 03 — Connect the selected environment
client = None
mongodb_uri = None
if USE_ATLAS:
    try:
        mongodb_uri = getpass("Atlas driver URI (hidden): ")
        client = MongoClient(
            mongodb_uri, tls=True, tlsInsecure=False, tz_aware=True,
            server_api=ServerApi("1", strict=True, deprecation_errors=True),
            serverSelectionTimeoutMS=10000, timeoutMS=10000,
        )
        client.admin.command("ping")
        print("MongoDB responded to ping.")
    except Exception:
        if client is not None:
            client.close()
        raise RuntimeError(
            "Atlas connection failed. Check the runtime IP rule, database "
            "credentials, driver URI, deployment state, and DNS/TLS. "
            "Keep certificate verification enabled; local mode is available."
        ) from None
    finally:
        mongodb_uri = None
else:
    client = mongomock.MongoClient(tz_aware=True)
    print("Local aggregation practice. Server validation will not execute.")

database = client[database_name]
tickets = database["tickets"]

## 1. Read the Four Tickets Before Querying

Each document is one ticket. `events` is a list of events **inside** that ticket.
An empty list is valid. A resolved ticket still exists but does not belong in an
active workload count. In this course, active means `new`, `open`, or `in_progress`.

Predict the active IDs and the number of active tickets in each category from
the fixture. This gives us a check independent of our aggregation syntax.

In [ ]:
#@title Cell 04a — Supplied definition of the four-ticket fixture
fixture = [
    {"ticket_id": 1001, "category": "streetlight", "status": "open",
     "priority": "urgent", "subject": "Dark streetlight", "assignee_id": 201,
     "opened_at": datetime(2026, 2, 1, tzinfo=timezone.utc),
     "events": [{"type": "created"}, {"type": "assigned"}]},
    {"ticket_id": 1002, "category": "sanitation", "status": "in_progress",
     "priority": "high", "subject": "Missed pickup", "assignee_id": 202,
     "opened_at": datetime(2026, 2, 2, tzinfo=timezone.utc),
     "events": [{"type": "created"}]},
    {"ticket_id": 1003, "category": "streetlight", "status": "resolved",
     "priority": "urgent", "subject": "Lamp repaired", "assignee_id": 201,
     "opened_at": datetime(2026, 2, 3, tzinfo=timezone.utc),
     "events": [{"type": "created"}, {"type": "resolved"}]},
    {"ticket_id": 1004, "category": "streetlight", "status": "new",
     "priority": "low", "subject": "Flickering lamp", "assignee_id": None,
     "opened_at": datetime(2026, 2, 4, tzinfo=timezone.utc), "events": []},
]


In [ ]:
#@title Cell 04 — Load only this run’s four-ticket collection
# Reset only this notebook's collection when repeating the fixture cell.
tickets.create_index("ticket_id", unique=True)
tickets.delete_many({})
tickets.insert_many(fixture)
pprint(list(tickets.find({}, {"_id": 0, "ticket_id": 1, "category": 1, "status": 1})))

## 2. Build a Summary One Stage at a Time

`$match` selects input documents. `$group` creates one output for each distinct
group key. `$sum: 1` adds one for each input reaching that group, regardless of
how many documents existed before earlier stages.

The dollar sign in `"$category"` means read that field's value. The string
`"category"` without the dollar would group every input under one constant label.
`$project` chooses the final shape, and `$sort` makes display order predictable.

In [ ]:
#@title Cell 05 — Trace the summary pipeline
active_filter = {"status": {"$in": ["new", "open", "in_progress"]}}

pipeline = [
    {"$match": active_filter},  # Input/output: one ticket per document.
    {"$group": {"_id": "$category", "active_count": {"$sum": 1}}},
    {"$project": {"_id": 0, "category": "$_id", "active_count": 1}},
    {"$sort": {"category": 1}},  # Output: one summary per category.
]

for end in range(1, len(pipeline) + 1):
    result = list(tickets.aggregate(pipeline[:end]))
    print("After", list(pipeline[end - 1])[0], ":", len(result), "documents")
    pprint(result)

### Your Change: Urgent Work and Newest Request

Add two outputs to the existing `$group` and expose them in `$project`:

```python
"urgent_count": {"$sum": {"$cond": [{"$eq": ["$priority", "urgent"]}, 1, 0]}},
"newest_opening": {"$max": "$opened_at"}
```

`$cond` chooses 1 or 0 per input ticket. `$max` keeps the largest date. Explain why
the resolved urgent ticket must not contribute to this **active** summary. Run
the modified pipeline and check streetlight by looking back at its ticket IDs.
Keep the active-count result when you add the two new fields.

In [ ]:
#@title Cell 06 — Your change: add urgent count and newest opening
# Edit these stages with the two expressions above, then rerun this cell.
my_pipeline = [
    {"$match": active_filter},
    {"$group": {"_id": "$category", "active_count": {"$sum": 1}}},
    {"$project": {"_id": 0, "category": "$_id", "active_count": 1}},
    {"$sort": {"category": 1}},
]
my_result = list(tickets.aggregate(my_pipeline))
pprint(my_result)

# An independent count checks the original report, not the new fields yet.
streetlight_count = tickets.count_documents({**active_filter, "category": "streetlight"})
print("Independent active streetlight count:", streetlight_count)
assert streetlight_count == 2

## 3. Discover a Counting Trap

Suppose we insert `$unwind: "$events"` before grouping. It produces one document
per array element. A ticket with two events now contributes twice; a ticket with
an empty array disappears unless we request preservation.

Predict the ticket IDs in the output. The total number of documents can
accidentally match the ticket count even when the report is wrong.

### Trace the records before this experiment

![Record relationships for Week 11](https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/main/week_11/illustrations/lesson_relationships.png)

The [complete data reference](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_11/DATA_REFERENCE.md) provides the same relationships as readable text.


In [ ]:
#@title Cell 07 — Compare ticket rows with unwound event rows
event_rows = list(tickets.aggregate([
    {"$match": active_filter},
    {"$unwind": "$events"},
    {"$project": {"_id": 0, "ticket_id": 1, "event_type": "$events.type"}},
    {"$sort": {"ticket_id": 1, "event_type": 1}},
]))
pprint(event_rows)
print("Active ticket IDs:", [row["ticket_id"] for row in tickets.find(active_filter)])
print("Unwound row IDs:", [row["ticket_id"] for row in event_rows])

For a ticket count, group the tickets without unwinding. For an **event** count,
unwinding is appropriate, but label the report as events. Preserving empty arrays
does not undo duplication for tickets with multiple events. SQL joins can cause
the same change of grain; remember the ticket-to-event relationship from Week 2.


## Explain today’s aggregation result

In the following explanation cell:

- Give the streetlight active count, urgent count and newest opening date. Identify the source tickets.
- Explain why the unwound rows count events rather than tickets, even when the totals happen to match.

**Validation is not needed for this lab. It remains optional reference; no next-week assignment is implied. Today, finish the aggregation work and submit your saved Colab copy’s sharing link.**


## Your Explanation

Write today’s aggregation explanation here. Keep it in this notebook. Validation is not part of this lab’s submission.


## Validation — not needed for this lab

**Validation is not needed for this lab. It remains optional reference; no next-week assignment is implied. Today, finish the aggregation work and submit your saved Colab copy’s sharing link.**

Expand this heading using its arrow when we return to validation. The section starts collapsed. Collapsing only hides cells, so the `RUN_VALIDATION` switch is also `False` to skip their execution today. Next week, rerun setup and load the four-ticket fixture, set the switch to `True`, then run this section in order.


In [ ]:
#@title Cell 08 — Optional validation switch
# Validation is not needed for this lab. Leave False today.
RUN_VALIDATION = False
print("Validation enabled for next week." if RUN_VALIDATION else "Validation is not needed for this lab; skipped today.")


### 4. Require Valid Statuses Without Requiring Every Field

A validator applies to writes. It does not summarize data or make a query faster.
This schema requires an integer ticket ID, an allowed status, and a subject.
Other fields may exist without being required. That is controlled flexibility.

The validator below is a MongoDB command document. It is not Python's type system
and is not identical to standard JSON Schema. BSON adds types such as `date`.

In [ ]:
#@title Cell 09 — Optional status validator
if RUN_VALIDATION:
    validator = {
        "$jsonSchema": {
            "bsonType": "object",
            "required": ["ticket_id", "status", "subject"],
            "properties": {
                "ticket_id": {"bsonType": ["int", "long"]},
                "status": {"enum": ["new", "open", "in_progress", "resolved", "closed"]},
                "subject": {"bsonType": "string"},
            },
        }
    }

    if USE_ATLAS:
        database.command({"collMod": "tickets", "validator": validator,
                          "validationLevel": "strict", "validationAction": "error"})
        print("MongoDB now checks new inserts and updates.")
    else:
        print("Validator defined but NOT installed: mongomock has no server validation.")
else:
    print("Validation is not needed for this lab; this step is skipped today.")


### Your Change: Opening Date

Require `opened_at`, and give it `{"bsonType": "date"}` in `properties`. Edit the
validator cell and rerun it. All four baseline tickets already have BSON dates.

The next test cell includes an allowed date, a date written as text, and a
missing date. A string that looks like a date is not a BSON date. The text and
missing cases should be rejected only after you add and reinstall your date rule.

Installing validation does not retroactively delete or repair existing invalid
documents. Inspect existing records before tightening a live collection. This
fresh practice fixture avoids an accidental migration of somebody else's data.

In [ ]:
#@title Cell 10 — Optional date validator test
if RUN_VALIDATION:
    valid_test = {"ticket_id": 1099, "status": "new", "subject": "Validation test",
                  "opened_at": datetime(2026, 2, 5, tzinfo=timezone.utc)}
    invalid_tests = [
        ("Invalid status", {**valid_test, "ticket_id": 1100, "status": "almost_done"}),
    ]
    date_rule_added = (
        "opened_at" in validator["$jsonSchema"]["required"]
        and validator["$jsonSchema"]["properties"].get("opened_at") == {"bsonType": "date"}
    )
    if date_rule_added:
        invalid_tests += [
            ("Date stored as text", {**valid_test, "ticket_id": 1101,
                                    "opened_at": "2026-02-05T00:00:00Z"}),
            ("Missing date", {"ticket_id": 1102, "status": "new", "subject": "No date"}),
        ]
    else:
        print("Date rule not enabled yet. Edit and rerun the validator cell, then this test.")

    if USE_ATLAS:
        try:
            # These IDs belong only to this test, so repeating it starts cleanly.
            tickets.delete_many({"ticket_id": {"$in": [1099, 1100, 1101, 1102]}})
            tickets.insert_one(valid_test)
            print("Valid insert accepted:", tickets.count_documents({"ticket_id": 1099}))
            for label, document in invalid_tests:
                try:
                    tickets.insert_one(document)
                except OperationFailure as error:
                    if error.code != 121:
                        raise
                    print(label + " rejected. MongoDB error code:", error.code)
                else:
                    raise AssertionError(label + " was accepted. Inspect the installed validator.")
        finally:
            tickets.delete_many({"ticket_id": {"$in": [1099, 1100, 1101, 1102]}})
    else:
        print("Trace to interpret, NOT a result from this runtime:")
        print("Valid insert accepted: 1")
        for label, document in invalid_tests:
            print(label + " rejected. MongoDB error code: 121")
else:
    print("Validation is not needed for this lab; this step is skipped today.")


### Optional reference: explaining validation

After completing the opening-date rule, explain the text-date and missing-date outcomes. State whether you executed MongoDB validation or interpreted the supplied local trace, and name one rule the validator does not enforce. Keep that explanation in this same notebook.


## Cleanup

The next cell deletes only `tickets` in the uniquely named practice database.
Run it when finished and remove the temporary Atlas IP entry in the dashboard.
A closed Python connection does not stop or delete an Atlas cluster. Do not delete
a shared cluster or another project's data. Do not put project data in this
practice collection.

In [ ]:
#@title Cell 11 — Clean up this run only
database.drop_collection("tickets")
client.close()
print("Removed this run's practice collection and closed its client.")

## Submit the link to your completed Colab copy

Save today’s aggregation code, outputs and explanation in your own Colab copy. Choose **File > Save**, then **Share** and give the instructor **viewer access**. Copy the sharing link and paste it into the Brightspace Lab 1 submission.

Submit **your saved copy’s Colab link**, not the starter notebook link. **Do not upload an `.ipynb` file to Brightspace.** If you completed the work in local Jupyter, upload the completed notebook to Colab, save it, and submit its sharing link.

The collapsed validation section is not needed for this lab; we will cover it next week.


## Further Reading

- [MongoDB aggregation pipeline](https://www.mongodb.com/docs/manual/core/aggregation-pipeline/)
- [Unwind behavior](https://www.mongodb.com/docs/manual/reference/operator/aggregation/unwind/)
- [Schema validation](https://www.mongodb.com/docs/manual/core/schema-validation/)
- [Atlas IP access list](https://www.mongodb.com/docs/atlas/security/ip-access-list/)

Course prose: CC BY-NC-SA 4.0. Code: MIT. Synthetic fixture: CC0.